# Phase 2 — BiLSTM magnitude regression, AdamW and Optuna

Only approved timestamp-fixed v2 catalog data. Next **ML catalog event** magnitude is an exploratory retrospective target, not operational EEW. No CNN/attention/waveforms. Phase 1 artifacts stay read-only.

Tune clean A with 15 TPE trials, validation MAE, median pruning. Fresh A/B/C/D fits share the selected settings, seed, loss, optimizer and stopping criteria. Validation selects one winner; a persisted selection lock precedes all test-label value access. The monolithic .pt is memory-mapped; only train/validation storages are copied before selection. Checkpointed epochs, SQLite study/sampler state and completed outputs support resumption on the same software/device.

[Graves & Schmidhuber (2005), Sections II–III](https://www.cs.toronto.edu/~graves/nn_2005.pdf) motivates bidirectional segment processing and BPTT. Both directions see only the historical window; standard PyTorch cells are not an exact paper reproduction. Catalog references and sampling adaptations are documented in the report. Training implementations and all executable experiment code are here.


In [1]:
from pathlib import Path
import os,sys,json,hashlib,time,random,pickle,ast,gc
from datetime import datetime,timezone
ROOT=Path.cwd().resolve()
if ROOT.name!="bilstm new updated":ROOT=ROOT/"bilstm new updated"
sys.path.insert(0,str(ROOT/".training-runtime"))
os.environ["CUBLAS_WORKSPACE_CONFIG"]=":4096:8"
import numpy as np
import pandas as pd
import torch
from torch import nn
import optuna
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
DATA=ROOT/"dataset"/"v2_timestamp_fixed"
AUG=ROOT/"augmentation_experiment"/"v2_timestamp_fixed"
OUT=ROOT/"phase2_bilstm_results"
for folder in [OUT,OUT/"trials",OUT/"variants",OUT/"plots"]:folder.mkdir(parents=True,exist_ok=True)
SEED=42;MAX_EPOCHS=50;PATIENCE=8;HUBER_DELTA=1.0;CLIP=1.0
torch.set_num_threads(4)
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.deterministic=True;torch.backends.cudnn.benchmark=False
torch.backends.cuda.matmul.allow_tf32=False;torch.backends.cudnn.allow_tf32=False
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
def seed_all():
    random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(SEED)
def sha(path):
    h=hashlib.sha256()
    with Path(path).open("rb") as f:
        for b in iter(lambda:f.read(1024*1024),b""):h.update(b)
    return h.hexdigest()
def content_sha(a):
    if isinstance(a,torch.Tensor):a=a.detach().cpu().contiguous().numpy()
    a=np.ascontiguousarray(a)
    h=hashlib.sha256();h.update(str(a.dtype).encode());h.update(str(a.shape).encode());h.update(a.tobytes())
    return h.hexdigest()
def replace_with_retry(tmp,path):
    # Windows indexing/antivirus can briefly hold an existing checkpoint file.
    for attempt in range(30):
        try:
            tmp.replace(path)
            return
        except PermissionError:
            if attempt==29:raise
            time.sleep(min(.1*(attempt+1),.5))
def save_json(path,value):
    path=Path(path);tmp=path.with_suffix(path.suffix+".tmp")
    tmp.write_text(json.dumps(value,indent=2,allow_nan=False),encoding="utf-8");replace_with_retry(tmp,path)
def save_torch(path,value):
    path=Path(path);tmp=path.with_suffix(path.suffix+".tmp");torch.save(value,tmp);replace_with_retry(tmp,path)
protected={}
for base,registry_name,key in [(DATA,"catalog_artifact_hashes.json","artifacts"),(AUG,"artifact_hashes.json","outputs")]:
    registry=json.loads((base/registry_name).read_text())
    for name,expected in registry[key].items():
        actual=sha(base/name);assert actual==expected,f"Source changed: {name}"
        protected[str((base/name).relative_to(ROOT))]=actual
dataset_hash=sha(DATA/"bilstm_catalog_seq20.pt")
ds_validation=json.loads((DATA/"dataset_validation.json").read_text())
aug_validation=json.loads((AUG/"validation_results.json").read_text())
sampler_manifest=json.loads((AUG/"sampler_manifest.json").read_text())
assert ds_validation["training_ready"] and aug_validation["training_ready"] and sampler_manifest["training_ready"]
assert not any(c["status"]=="FAIL" for c in ds_validation["checks"].values())
assert aug_validation["overall_status"]=="PASS"
assert sampler_manifest["source_sha256"]==dataset_hash
target_definition=json.loads((DATA/"target_definition.json").read_text())
scaling=json.loads((DATA/"catalog_train_scalers.json").read_text())
aug_config=json.loads((AUG/"augmentation_config.json").read_text())
assert target_definition["target_transform"]=={"name":"identity","offset":0.0,"scale":1.0}
# Mapped test storages are neither indexed, copied, inspected nor used during model selection.
mapped=torch.load(DATA/"bilstm_catalog_seq20.pt",map_location="cpu",weights_only=True,mmap=True)
train_X=mapped["X_train"].clone();train_y=mapped["y_reg_train"].clone()
train_binary=mapped["y_binary_train"].clone()
val_X=mapped["X_val"].clone();val_y=mapped["y_reg_val"].clone()
del mapped
assert tuple(train_X.shape)==(4973,20,10) and tuple(val_X.shape)==(1065,20,10)
assert all(torch.isfinite(t).all() for t in [train_X,train_y,val_X,val_y])
# Read only required training provenance fields; target magnitudes are not read from this manifest.
parent_manifest=pd.read_csv(DATA/"split_manifest.csv",usecols=["split","sample_index","input_event_ids","target_event_id"])
train_manifest=parent_manifest.loc[parent_manifest.split.eq("train")].reset_index(drop=True)
input_ids=np.array([json.loads(v) for v in train_manifest.input_event_ids])
unique_ids=np.unique(input_ids);id_rows={v:i for i,v in enumerate(unique_ids)}
history_rows=np.array([[id_rows[v] for v in row] for row in input_ids])
# Reuse approved functions verbatim by extracting definitions, never executing the preprocessing notebook.
aug_notebook=json.loads((ROOT/"phase1b_imbalance_noise_preprocessing.ipynb").read_text())
function_names={"draw_parent_indices","coordinate_noise","diagnostic_variant","prepare_training_epoch"}
function_nodes=[]
for cell in aug_notebook["cells"]:
    if cell["cell_type"]=="code":
        function_nodes.extend(node for node in ast.parse("".join(cell["source"])).body
                              if isinstance(node,ast.FunctionDef) and node.name in function_names)
assert {node.name for node in function_nodes}==function_names
consumer_source=ast.unparse(ast.Module(body=function_nodes,type_ignores=[]))
consumer_hash=hashlib.sha256(consumer_source.encode()).hexdigest()
with np.load(AUG/"sampling_plan.npz",allow_pickle=False) as plan:
    approved_weights=plan["weights"].copy()
    preview_indices={v:plan["parent_rows_"+v].copy() for v in "ABCD"}
scope={"np":np,"CONFIG":aug_config,"SEED":SEED,"N":len(train_y),"probabilities":approved_weights/approved_weights.sum(),
       "X":train_X.numpy(),"y":train_y.numpy(),"binary":train_binary.numpy(),
       "unique_history_ids":unique_ids,"history_rows":history_rows,
       "mean":np.array(scaling["scaler"]["mean"]),"scale":np.array(scaling["scaler"]["scale"]),"source_ok":True}
exec(compile(ast.Module(body=function_nodes,type_ignores=[]),"<approved-v2-augmentation-consumers>","exec"),scope)
epoch_data=scope["prepare_training_epoch"]
approved_previews={r["variant"]:r for r in json.loads((AUG/"distribution_comparison.json").read_text())}
for variant in "ABCD":
    parents,x,y,b=epoch_data(variant,0)
    assert np.array_equal(parents,preview_indices[variant])
    assert content_sha(x)==approved_previews[variant]["X_content_sha256"]
    assert np.array_equal(y,train_y.numpy()[parents]) and np.array_equal(b,train_binary.numpy()[parents])
del parents,x,y,b
PROTOCOL={"version":"phase2_v1","dataset_version":"v2_timestamp_fixed","dataset_sha256":dataset_hash,
 "dataset_run_id":ds_validation["run_id"],"augmentation_run_id":aug_config["run_id"],
 "consumer_source_sha256":consumer_hash,"seed":SEED,"max_epochs":50,"patience":8,"gradient_clip":1.0,
 "loss":"Huber","huber_delta":1.0,"optimizer":"AdamW_all_weights_and_biases",
 "search_trials":15,"tuning_variant":"A","objective":"validation_MAE_original_ML",
 "sampler":"TPE_seed42_startup5","pruner":"Median_startup5_warmup8",
 "selection":"minimum_validation_MAE_then_variant_name","data_order":"approved_chronological_parent_draws_no_extra_shuffle",
 "torch":str(torch.__version__),"optuna":str(optuna.__version__),"numpy":np.__version__,
 "device":str(DEVICE),"gpu":torch.cuda.get_device_name(0) if DEVICE.type=="cuda" else None,
 "deterministic_algorithms":True,"TF32":False,"test_values_accessed_before_selection":False}
RUN_ID=hashlib.sha256(json.dumps(PROTOCOL,sort_keys=True).encode()).hexdigest()[:16]
PROTOCOL["run_id"]=RUN_ID
if (OUT/"protocol.json").exists():assert json.loads((OUT/"protocol.json").read_text())==PROTOCOL,"Protocol differs; do not overwrite prior run."
save_json(OUT/"protocol.json",PROTOCOL)
train_gpu=train_X.to(DEVICE);val_gpu=val_X.to(DEVICE)
print(json.dumps({"run_id":RUN_ID,"device":str(DEVICE),"gpu":PROTOCOL["gpu"],"approved_consumers_verified":4,
                  "train_shape":list(train_X.shape),"validation_shape":list(val_X.shape),"test_values_read":False}))


{"run_id": "1bd0faa814c714ba", "device": "cuda", "gpu": "NVIDIA GeForce RTX 3050 Laptop GPU", "approved_consumers_verified": 4, "train_shape": [4973, 20, 10], "validation_shape": [1065, 20, 10], "test_values_read": false}


## Model and resumable training

Final-layer forward/backward states are concatenated as `h_n[-2]` and `h_n[-1]`; no last-time-step backward shortcut. AdamW receives every trainable weight and bias. Huber δ=1, gradient clipping 1, maximum 50 epochs, patience 8 on validation MAE. Head dropout is always applied; inter-layer dropout applies only with two LSTM layers.

Each fit resets seed 42 and starts fresh unless resuming its own matching checkpoint. The saved epoch state includes optimizer and torch CPU/CUDA RNG states. Histories record both augmented training objective and clean-training evaluation, so overfitting gaps use comparable inference-mode predictions.


In [2]:
class BiLSTMRegressor(nn.Module):
    def __init__(self,hidden_size,layers,dropout):
        super().__init__()
        self.lstm=nn.LSTM(10,hidden_size,num_layers=layers,batch_first=True,bidirectional=True,
                          dropout=dropout if layers>1 else 0.)
        self.dropout=nn.Dropout(dropout)
        self.head=nn.Linear(2*hidden_size,1)
    def forward(self,x):
        _,(h,_) = self.lstm(x)
        return self.head(self.dropout(torch.cat([h[-2],h[-1]],dim=1))).squeeze(-1)
def model_config(cfg):return {k:cfg[k] for k in ["hidden_size","layers","dropout"]}
def state_hash(model):
    h=hashlib.sha256()
    for name,tensor in model.state_dict().items():
        h.update(name.encode());h.update(tensor.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()
def sync():
    if DEVICE.type=="cuda":torch.cuda.synchronize()
def predict(model,x,batch=256):
    model.eval();out=[]
    with torch.inference_mode():
        for start in range(0,len(x),batch):out.append(model(x[start:start+batch].to(DEVICE)).cpu())
    return torch.cat(out).numpy()
def metric_values(actual,pred):
    actual=np.asarray(actual,dtype=np.float64);pred=np.asarray(pred,dtype=np.float64)
    residual=pred-actual;absolute=np.abs(residual);high=actual>=4.5
    return {"mae":float(absolute.mean()),"rmse":float(np.sqrt(np.mean(residual**2))),
            "r2":float(1-np.sum(residual**2)/np.sum((actual-actual.mean())**2)),
            "huber":float(np.where(absolute<=HUBER_DELTA,.5*residual**2,HUBER_DELTA*(absolute-.5*HUBER_DELTA)).mean()),
            "high_count":int(high.sum()),"high_mae":float(absolute[high].mean()) if high.any() else None,
            "high_underprediction_rate":float((residual[high]<0).mean()) if high.any() else None,
            "high_bias":float(residual[high].mean()) if high.any() else None}
def reload_best(path):
    ck=torch.load(path,map_location="cpu",weights_only=True)
    assert ck["run_id"]==RUN_ID and ck["dataset_sha256"]==dataset_hash
    m=BiLSTMRegressor(**ck["model_config"]).to(DEVICE);m.load_state_dict(ck["model_state"]);m.eval()
    return m,ck
# Meaningful architecture/optimizer coverage check on historical training inputs only.
seed_all();probe=BiLSTMRegressor(32,2,.2).to(DEVICE).eval()
with torch.no_grad():
    seq,(hidden,_) = probe.lstm(train_gpu[:3])
    assert torch.allclose(seq[:,-1,:32],hidden[-2],atol=1e-6)
    assert torch.allclose(seq[:,0,32:],hidden[-1],atol=1e-6)
probe_opt=torch.optim.AdamW(probe.parameters(),lr=.001)
assert {id(p) for g in probe_opt.param_groups for p in g["params"]}=={id(p) for p in probe.parameters()}
probe.train();nn.HuberLoss(delta=1.)(probe(train_gpu[:3]),train_y[:3].to(DEVICE)).backward()
assert all(p.requires_grad and p.grad is not None and torch.isfinite(p.grad).all() for p in probe.parameters())
del probe,probe_opt
if DEVICE.type=="cuda":torch.cuda.empty_cache()

def fit_model(cfg,variant,folder,trial=None):
    folder=Path(folder);folder.mkdir(parents=True,exist_ok=True)
    summary_path=folder/"fit_summary.json";last_path=folder/"last.pt";best_path=folder/"best.pt"
    fit_key={"run_id":RUN_ID,"variant":variant,"hyperparameters":cfg}
    if summary_path.exists():
        summary=json.loads(summary_path.read_text())
        assert summary["fit_key"]==fit_key
        if summary["status"]=="PRUNED":raise optuna.TrialPruned()
        assert summary["status"]=="COMPLETE" and sha(best_path)==summary["best_checkpoint_sha256"]
        return summary
    seed_all()
    model=BiLSTMRegressor(**model_config(cfg)).to(DEVICE)
    initial_hash=state_hash(model)
    optimizer=torch.optim.AdamW(model.parameters(),lr=cfg["learning_rate"],weight_decay=cfg["weight_decay"])
    assert all(p.requires_grad for p in model.parameters())
    assert {id(p) for g in optimizer.param_groups for p in g["params"]}=={id(p) for p in model.parameters()}
    loss_fn=nn.HuberLoss(delta=HUBER_DELTA)
    history=[];best_mae=float("inf");best_epoch=0;stale=0;elapsed=0.;start_epoch=1
    parameter_count=sum(p.numel() for p in model.parameters())
    if last_path.exists():
        last=torch.load(last_path,map_location="cpu",weights_only=True)
        assert last["fit_key"]==fit_key
        model.load_state_dict(last["model_state"]);optimizer.load_state_dict(last["optimizer_state"])
        history=last["history"];best_mae=last["best_mae"];best_epoch=last["best_epoch"]
        stale=last["stale"];elapsed=last["elapsed_seconds"];start_epoch=last["epoch"]+1
        assert initial_hash==last["initialization_sha256"]
        torch.set_rng_state(last["cpu_rng"])
        if DEVICE.type=="cuda":torch.cuda.set_rng_state_all(last["cuda_rng"])
    pruned=False
    for epoch in range(start_epoch,MAX_EPOCHS+1):
        if stale>=PATIENCE:break
        sync();epoch_start=time.perf_counter()
        parents,x_epoch,y_epoch,_=epoch_data(variant,epoch-1)
        assert np.array_equal(y_epoch,train_y.numpy()[parents])
        x_device=torch.from_numpy(x_epoch).to(DEVICE);y_device=torch.from_numpy(y_epoch).to(DEVICE)
        model.train();loss_sum=torch.zeros((),device=DEVICE)
        for start in range(0,len(parents),cfg["batch_size"]):
            end=min(start+cfg["batch_size"],len(parents))
            optimizer.zero_grad(set_to_none=True)
            predictions=model(x_device[start:end]);loss=loss_fn(predictions,y_device[start:end])
            loss.backward();nn.utils.clip_grad_norm_(model.parameters(),CLIP);optimizer.step()
            loss_sum+=loss.detach()*(end-start)
        epoch_loss=float(loss_sum.item()/len(parents))
        train_pred=predict(model,train_gpu);val_pred=predict(model,val_gpu)
        train_metrics=metric_values(train_y.numpy(),train_pred);val_metrics=metric_values(val_y.numpy(),val_pred)
        assert np.isfinite([epoch_loss,train_metrics["mae"],val_metrics["mae"]]).all()
        improved=val_metrics["mae"]<best_mae
        if improved:
            best_mae=val_metrics["mae"];best_epoch=epoch;stale=0
            save_torch(best_path,{"run_id":RUN_ID,"model_state":{k:v.detach().cpu().clone() for k,v in model.state_dict().items()},
              "model_config":model_config(cfg),"hyperparameters":cfg,"dataset_version":"v2_timestamp_fixed",
              "dataset_sha256":dataset_hash,"augmentation_variant":variant,"augmentation_run_id":aug_config["run_id"],
              "consumer_source_sha256":consumer_hash,"target_definition":target_definition,"best_epoch":epoch,
              "validation_mae":best_mae,"seed":SEED,"parameter_count":parameter_count,
              "initialization_sha256":initial_hash,"optimizer":"AdamW_all_trainable_weights_and_biases",
              "loss":"Huber_delta1","fit_key":fit_key})
        else:stale+=1
        sync();elapsed+=time.perf_counter()-epoch_start
        history.append({"epoch":epoch,"training_augmented_huber":epoch_loss,
            **{"train_"+k:v for k,v in train_metrics.items()},**{"val_"+k:v for k,v in val_metrics.items()},
            "unique_parents":int(len(np.unique(parents))),"repeated_draws":int(len(parents)-len(np.unique(parents))),
            "parent_indices_sha256":content_sha(parents),"effective_high_fraction":float((y_epoch>=4.5).mean()),
            "elapsed_seconds":elapsed})
        save_json(folder/"history.json",history)
        save_torch(last_path,{"fit_key":fit_key,"model_state":model.state_dict(),"optimizer_state":optimizer.state_dict(),
            "history":history,"epoch":epoch,"best_mae":best_mae,"best_epoch":best_epoch,"stale":stale,
            "elapsed_seconds":elapsed,"initialization_sha256":initial_hash,
            "cpu_rng":torch.get_rng_state(),"cuda_rng":torch.cuda.get_rng_state_all() if DEVICE.type=="cuda" else []})
        save_json(OUT/"progress.json",{"stage":"optuna" if trial is not None else "controlled_fit",
            "trial":trial.number if trial is not None else None,"variant":variant,"epoch":epoch,
            "best_epoch":best_epoch,"best_validation_mae":best_mae,"fit_elapsed_seconds":elapsed})
        del x_device,y_device
        if trial is not None:
            trial.report(val_metrics["mae"],step=epoch)
            if trial.should_prune():pruned=True;break
    assert best_path.exists() and history
    summary={"fit_key":fit_key,"status":"PRUNED" if pruned else "COMPLETE","best_epoch":best_epoch,
       "best_validation_mae":best_mae,"epochs_run":len(history),"training_seconds":elapsed,
       "parameter_count":parameter_count,"initialization_sha256":initial_hash,"best_checkpoint_sha256":sha(best_path)}
    save_json(summary_path,summary)
    del model,optimizer
    if DEVICE.type=="cuda":torch.cuda.empty_cache()
    if pruned:raise optuna.TrialPruned()
    return summary
print("BiLSTM hidden-state and AdamW coverage checks passed; resumable fit implementation ready.")


BiLSTM hidden-state and AdamW coverage checks passed; resumable fit implementation ready.


## Validation-only TPE search and four controlled fits

Fifteen terminal trials (completed or pruned), clean A only. Median pruning starts after five completed trials and eight epochs. The study and sampler RNG are persisted; interrupted fits resume their own epoch checkpoints. All four final models start from identical seeded fresh initialization, not the tuning checkpoint. Completed fits are reused only when configuration and source hashes match.


In [3]:
optuna.logging.set_verbosity(optuna.logging.WARNING)
sampler_path=OUT/"optuna_sampler.pkl"
if sampler_path.exists():
    with sampler_path.open("rb") as f:tpe=pickle.load(f) # Own locally generated sampler state only.
else:tpe=optuna.samplers.TPESampler(seed=SEED,n_startup_trials=5)
pruner=optuna.pruners.MedianPruner(n_startup_trials=5,n_warmup_steps=8,interval_steps=1)
study=optuna.create_study(study_name="clean_A_"+RUN_ID,storage="sqlite:///"+(OUT/"optuna_study.sqlite3").as_posix(),
                         direction="minimize",sampler=tpe,pruner=pruner,load_if_exists=True)
if "run_id" in study.user_attrs:assert study.user_attrs["run_id"]==RUN_ID
study.set_user_attr("run_id",RUN_ID);study.set_user_attr("test_values_accessed",False)
def persist_sampler():
    tmp=sampler_path.with_suffix(".tmp")
    with tmp.open("wb") as f:pickle.dump(study.sampler,f)
    replace_with_retry(tmp,sampler_path)
def choose_trial_config(trial):
    return {"hidden_size":trial.suggest_categorical("hidden_size",[32,64,128]),
            "layers":trial.suggest_int("layers",1,2),"dropout":trial.suggest_float("dropout",.1,.4),
            "learning_rate":trial.suggest_float("learning_rate",1e-4,3e-3,log=True),
            "weight_decay":trial.suggest_float("weight_decay",1e-6,1e-3,log=True),
            "batch_size":trial.suggest_categorical("batch_size",[32,64,128])}
terminal_states=(optuna.trial.TrialState.COMPLETE,optuna.trial.TrialState.PRUNED)
while sum(t.state in terminal_states for t in study.trials)<15:
    assert not (OUT/"selection_lock.json").exists(),"Selection already locked; tuning is prohibited."
    running=[t for t in study.trials if t.state==optuna.trial.TrialState.RUNNING]
    assert len(running)<=1
    trial=optuna.trial.Trial(study,running[0]._trial_id) if running else study.ask()
    cfg=choose_trial_config(trial);persist_sampler()
    try:
        result=fit_model(cfg,"A",OUT/"trials"/f"trial_{trial.number:03d}",trial=trial)
        study.tell(trial,result["best_validation_mae"])
        print(f"TPE {trial.number}: complete, best val MAE={result['best_validation_mae']:.4f}, epoch={result['best_epoch']}",flush=True)
    except optuna.TrialPruned:
        study.tell(trial,state=optuna.trial.TrialState.PRUNED)
        print(f"TPE {trial.number}: pruned",flush=True)
    persist_sampler()
study.trials_dataframe().to_csv(OUT/"optuna_trials.csv",index=False)
best_cfg=dict(study.best_trial.params)
best_hparams={"run_id":RUN_ID,"selection_basis":"clean A validation MAE only","best_trial":study.best_trial.number,
              "best_tuning_validation_mae":study.best_value,"parameters":best_cfg,
              "completed_trials":sum(t.state==optuna.trial.TrialState.COMPLETE for t in study.trials),
              "pruned_trials":sum(t.state==optuna.trial.TrialState.PRUNED for t in study.trials)}
save_json(OUT/"best_hyperparameters.json",best_hparams)
print("Selected shared hyperparameters:",json.dumps(best_cfg),flush=True)


TPE 0: complete, best val MAE=0.6156, epoch=17
TPE 1: complete, best val MAE=1.0773, epoch=29
TPE 2: complete, best val MAE=0.8973, epoch=28
TPE 3: complete, best val MAE=0.5078, epoch=7
TPE 4: complete, best val MAE=0.5619, epoch=2
TPE 5: complete, best val MAE=0.5073, epoch=2
TPE 6: complete, best val MAE=0.4791, epoch=8
TPE 7: pruned
TPE 8: complete, best val MAE=0.4783, epoch=14
TPE 9: pruned
TPE 10: complete, best val MAE=0.4865, epoch=12
TPE 11: complete, best val MAE=0.4866, epoch=10
TPE 12: complete, best val MAE=0.4883, epoch=1
TPE 13: pruned
TPE 14: pruned
Selected shared hyperparameters: {"hidden_size": 128, "layers": 2, "dropout": 0.3144753602375439, "learning_rate": 0.0004549047669323766, "weight_decay": 0.00012155233771053111, "batch_size": 128}


In [4]:
validation_rows=[];variant_summaries={};all_histories={};initial_hashes=[]
for variant in "ABCD":
    folder=OUT/"variants"/variant
    result=fit_model(best_cfg,variant,folder)
    variant_summaries[variant]=result
    initial_hashes.append(result["initialization_sha256"])
    history=json.loads((folder/"history.json").read_text());all_histories[variant]=history
    model,ck=reload_best(folder/"best.pt")
    val_pred=predict(model,val_gpu);train_pred=predict(model,train_gpu)
    vm=metric_values(val_y.numpy(),val_pred);tm=metric_values(train_y.numpy(),train_pred)
    assert abs(vm["mae"]-result["best_validation_mae"])<1e-7
    # A second independent checkpoint reload reproduces validation predictions.
    check_model,_=reload_best(folder/"best.pt")
    assert np.array_equal(val_pred,predict(check_model,val_gpu))
    np.savez_compressed(folder/"validation_predictions.npz",actual=val_y.numpy(),prediction=val_pred)
    best_row=history[result["best_epoch"]-1];last_row=history[-1]
    row={"variant":variant,**{"validation_"+k:v for k,v in vm.items()},
         "clean_train_mae":tm["mae"],"clean_train_rmse":tm["rmse"],"clean_train_huber":tm["huber"],
         "generalization_gap_mae":vm["mae"]-tm["mae"],"best_epoch":result["best_epoch"],
         "epochs_run":result["epochs_run"],"parameter_count":result["parameter_count"],
         "training_seconds":result["training_seconds"],
         "post_best_validation_mae_change":last_row["val_mae"]-best_row["val_mae"],
         "post_best_train_mae_change":last_row["train_mae"]-best_row["train_mae"],
         "late_overfitting_pattern":bool(last_row["val_mae"]>best_row["val_mae"]+.01 and last_row["train_mae"]<best_row["train_mae"]),
         "mean_unique_parents_per_epoch":float(np.mean([r["unique_parents"] for r in history])),
         "mean_repeated_draws_per_epoch":float(np.mean([r["repeated_draws"] for r in history])),
         "mean_effective_high_fraction":float(np.mean([r["effective_high_fraction"] for r in history])),
         "checkpoint_sha256":sha(folder/"best.pt")}
    validation_rows.append(row)
    del model,check_model
    print(f"Variant {variant}: validation MAE={vm['mae']:.4f}, RMSE={vm['rmse']:.4f}, best epoch={result['best_epoch']}",flush=True)
assert len(set(initial_hashes))==1,"All four fits must have identical fresh initial weights."
assert abs(validation_rows[0]["validation_mae"]-study.best_value)<1e-7,"Clean refit must reproduce the selected trial."
comparison=pd.DataFrame(validation_rows)
comparison.to_csv(OUT/"validation_comparison.csv",index=False)
# C and D use identical complete-window draws for each common epoch.
for c,d in zip(all_histories["C"],all_histories["D"]):
    assert c["parent_indices_sha256"]==d["parent_indices_sha256"]
winner_row=min(validation_rows,key=lambda r:(r["validation_mae"],r["variant"]))
winner=winner_row["variant"]
selection={"run_id":RUN_ID,"winner":winner,"criterion":"minimum validation MAE, deterministic alphabetical tie break",
 "selected_validation_mae":winner_row["validation_mae"],"checkpoint_sha256":winner_row["checkpoint_sha256"],
 "validation_comparison_sha256":sha(OUT/"validation_comparison.csv"),
 "dataset_sha256":dataset_hash,"test_values_accessed_during_selection":False}
if (OUT/"selection_lock.json").exists():assert json.loads((OUT/"selection_lock.json").read_text())==selection
else:save_json(OUT/"selection_lock.json",selection)
print("Validation selection locked:",winner,"; test evaluation may now run.",flush=True)


Variant A: validation MAE=0.4783, RMSE=0.6356, best epoch=14
Variant B: validation MAE=0.4772, RMSE=0.6350, best epoch=14
Variant C: validation MAE=0.4724, RMSE=0.6242, best epoch=12
Variant D: validation MAE=0.4727, RMSE=0.6245, best epoch=12
Validation selection locked: C ; test evaluation may now run.


## Winner-only holdout evaluation

The selection lock is immutable for this run. Test predictions are generated for that winner only, cached, and checked using independent NumPy and scikit-learn calculations. Mean/median baselines use clean training targets; the recent-event baseline reverses the saved training scaler for the last input magnitude, with no target information. No subsequent model or preprocessing selection uses test outcomes.


In [5]:
assert (OUT/"selection_lock.json").exists()
selection_hash=sha(OUT/"selection_lock.json")
test_cache=OUT/"selected_test_predictions.npz"
test_metrics_path=OUT/"selected_test_metrics.json"
if test_cache.exists():
    with np.load(test_cache,allow_pickle=False) as z:
        assert str(z["selection_sha256"])==selection_hash
        test_actual=z["actual"].copy();test_pred=z["prediction"].copy();recent_pred=z["recent_event_baseline"].copy()
    test_inference_reused=True
else:
    heldout=torch.load(DATA/"bilstm_catalog_seq20.pt",map_location="cpu",weights_only=True,mmap=True)
    test_X=heldout["X_test"].clone();test_y=heldout["y_reg_test"].clone();del heldout
    assert tuple(test_X.shape)==(1067,20,10)
    model,ck=reload_best(OUT/"variants"/winner/"best.pt")
    assert sha(OUT/"variants"/winner/"best.pt")==selection["checkpoint_sha256"]
    test_pred=predict(model,test_X);test_actual=test_y.numpy().copy()
    recent_pred=test_X[:,-1,0].numpy().astype(np.float64)*scaling["scaler"]["scale"][0]+scaling["scaler"]["mean"][0]
    np.savez_compressed(test_cache,actual=test_actual,prediction=test_pred,recent_event_baseline=recent_pred,
                        selection_sha256=np.array(selection_hash))
    del test_X,test_y,model
    test_inference_reused=False
test_metrics=metric_values(test_actual,test_pred)
assert np.isclose(test_metrics["mae"],mean_absolute_error(test_actual.astype(float),test_pred.astype(float)),rtol=0,atol=1e-12)
assert np.isclose(test_metrics["rmse"],np.sqrt(mean_squared_error(test_actual.astype(float),test_pred.astype(float))),rtol=0,atol=1e-12)
assert np.isclose(test_metrics["r2"],r2_score(test_actual.astype(float),test_pred.astype(float)),rtol=0,atol=1e-12)
baseline_predictions={"training_mean":np.full(len(test_actual),float(train_y.double().mean())),
                      "training_median":np.full(len(test_actual),float(np.median(train_y.numpy()))),
                      "recent_event_magnitude":recent_pred}
baseline_rows=[{"model":"BiLSTM_"+winner,**test_metrics}]
for name,prediction in baseline_predictions.items():baseline_rows.append({"model":name,**metric_values(test_actual,prediction)})
pd.DataFrame(baseline_rows).to_csv(OUT/"test_baseline_comparison.csv",index=False)
ranges=[(-np.inf,3.),(3.,4.),(4.,4.5),(4.5,5.),(5.,np.inf)]
range_labels=["<3","3–<4","4–<4.5","4.5–<5",">=5"]
range_rows=[]
for (lo,hi),label in zip(ranges,range_labels):
    mask=(test_actual>=lo)&(test_actual<hi)
    residual=test_pred[mask].astype(float)-test_actual[mask].astype(float)
    range_rows.append({"magnitude_range":label,"count":int(mask.sum()),
                      "mae":float(np.abs(residual).mean()) if mask.any() else None,
                      "rmse":float(np.sqrt(np.mean(residual**2))) if mask.any() else None,
                      "bias":float(residual.mean()) if mask.any() else None,
                      "underprediction_rate":float((residual<0).mean()) if mask.any() else None})
pd.DataFrame(range_rows).to_csv(OUT/"test_error_by_magnitude.csv",index=False)
residual=test_pred.astype(float)-test_actual.astype(float)
test_result={"run_id":RUN_ID,"selected_variant":winner,"selection_sha256":selection_hash,
 "metrics":test_metrics,"error_distribution":{"residual_mean":float(residual.mean()),"residual_std":float(residual.std()),
 "absolute_error_quantiles":dict(zip(["q50","q90","q95","q99"],np.quantile(np.abs(residual),[.5,.9,.95,.99]).tolist()))},
 "magnitude_ranges":range_rows,"baselines":baseline_rows[1:],"test_model_variants_evaluated":[winner],
 "independent_sklearn_verification":"PASS","prediction_cache_sha256":sha(test_cache),
 "baseline_fitting":"training targets only; recent-event uses last historical input and saved train-fitted inverse scaler"}
if test_metrics_path.exists():
    previous=json.loads(test_metrics_path.read_text());assert previous==test_result,"Cached holdout results differ."
else:save_json(test_metrics_path,test_result)
print("Winner-only test:",json.dumps(test_metrics),flush=True)


Winner-only test: {"mae": 0.5476705028056652, "rmse": 0.6696078672957082, "r2": -0.08986382358779799, "huber": 0.21490484945104285, "high_count": 38, "high_mae": 1.5217757036811428, "high_underprediction_rate": 1.0, "high_bias": -1.5217757036811428}


## Saved evidence and final verification

Curves distinguish clean-training inference metrics from the noisy/weighted optimization loss. A post-best decrease in clean-training MAE alongside a validation increase >0.01 is recorded as a late-training overfitting pattern, not proof that distribution shift is absent. Same-seed results establish reproducibility for this environment, not robustness across seeds.


In [6]:
for metric,label in [("huber","Huber loss"),("mae","MAE (ML units)")]:
    fig,axes=plt.subplots(2,2,figsize=(10,7))
    for ax,v in zip(axes.ravel(),"ABCD"):
        hist=all_histories[v];epochs=[r["epoch"] for r in hist]
        ax.plot(epochs,[r["train_"+metric] for r in hist],label="Clean training")
        ax.plot(epochs,[r["val_"+metric] for r in hist],label="Validation")
        if metric=="huber":
            ax.plot(epochs,[r["training_augmented_huber"] for r in hist],alpha=.5,ls=":",label="Training objective")
        ax.axvline(variant_summaries[v]["best_epoch"],color="gray",ls="--",label="Selected epoch")
        ax.set(title="Variant "+v,xlabel="Epoch",ylabel=label)
    axes[0,0].legend(fontsize=8);fig.tight_layout()
    fig.savefig(OUT/"plots"/("training_validation_"+metric+".png"),dpi=150);plt.close(fig)
fig,axes=plt.subplots(1,2,figsize=(10,4))
axes[0].scatter(test_actual,test_pred,s=12,alpha=.45)
lo=min(float(test_actual.min()),float(test_pred.min()));hi=max(float(test_actual.max()),float(test_pred.max()))
axes[0].plot([lo,hi],[lo,hi],"k--");axes[0].set(xlabel="Actual ML",ylabel="Predicted ML",title="Selected "+winner+" — test")
axes[1].hist(residual,bins=30);axes[1].axvline(0,color="black",ls="--")
axes[1].set(xlabel="Prediction minus actual (ML)",ylabel="Count",title="Test residual distribution")
fig.tight_layout();fig.savefig(OUT/"plots"/"selected_test_predictions_errors.png",dpi=150);plt.close(fig)
fig,ax=plt.subplots(figsize=(7,4))
ax.bar([r["magnitude_range"] for r in range_rows],[r["mae"] if r["mae"] is not None else 0 for r in range_rows])
for i,r in enumerate(range_rows):
    if r["mae"] is not None:ax.text(i,r["mae"],"n="+str(r["count"]),ha="center",va="bottom",fontsize=9)
ax.set(xlabel="Actual target magnitude",ylabel="Test MAE (ML units)",title="Selected model error by magnitude")
ax.margins(y=.2);fig.tight_layout();fig.savefig(OUT/"plots"/"test_error_by_magnitude.png",dpi=150);plt.close(fig)
# Independent reload of persisted validation predictions and metric table.
saved_comparison=pd.read_csv(OUT/"validation_comparison.csv")
for row in saved_comparison.itertuples():
    with np.load(OUT/"variants"/row.variant/"validation_predictions.npz",allow_pickle=False) as z:
        assert np.isclose(row.validation_mae,mean_absolute_error(z["actual"].astype(float),z["prediction"].astype(float)),atol=1e-12)
        assert np.isclose(row.validation_rmse,np.sqrt(mean_squared_error(z["actual"].astype(float),z["prediction"].astype(float))),atol=1e-12)
assert winner==min(validation_rows,key=lambda r:(r["validation_mae"],r["variant"]))["variant"]
assert sha(OUT/"selection_lock.json")==selection_hash
assert all(sha(ROOT/path)==expected for path,expected in protected.items())
trial_summaries=[json.loads((OUT/"trials"/f"trial_{t.number:03d}"/"fit_summary.json").read_text()) for t in study.trials if t.state in terminal_states]
tuning_seconds=sum(r["training_seconds"] for r in trial_summaries)
controlled_seconds=sum(r["training_seconds"] for r in variant_summaries.values())
verification={"run_id":RUN_ID,"status":"PASS",
 "corrected_v2_only":True,"dataset_sha256":dataset_hash,"original_phase1_artifacts_unchanged":True,
 "approved_consumers_verbatim_sha256":consumer_hash,"approved_preview_replay":"PASS",
 "bidirectional_final_hidden_states":"PASS","all_weights_and_biases_trainable_AdamW_covered":"PASS",
 "gradient_backpropagation_all_parameters":"PASS","gradient_clip":CLIP,"huber_delta":HUBER_DELTA,
 "optuna_terminal_trials":len(trial_summaries),"optuna_complete":best_hparams["completed_trials"],"optuna_pruned":best_hparams["pruned_trials"],
 "study_resume_storage":"optuna_study.sqlite3","sampler_rng_persisted":True,"epoch_optimizer_rng_resume":True,
 "four_separate_fits":True,"same_fresh_initialization_sha256":initial_hashes[0],
 "clean_refit_reproduces_best_trial":"PASS","all_best_checkpoints_reloaded_validation_exact":"PASS",
 "validation_predictions_independently_verified":"PASS","test_metrics_independently_verified":"PASS",
 "selected_before_test_values_accessed":True,"selection_lock_sha256":selection_hash,"test_model_variants_evaluated":[winner],
 "tuning_compute_seconds":tuning_seconds,"controlled_fit_compute_seconds":controlled_seconds,
 "total_fit_compute_seconds":tuning_seconds+controlled_seconds,"device":PROTOCOL["gpu"] or "CPU",
 "operational_catalog_availability":"UNKNOWN","magnitude_calibration":"UNKNOWN","CNN_hybrid_attention_runs":0}
save_json(OUT/"verification.json",verification)
val_table=[]
for r in validation_rows:
    val_table.append(f"| {r['variant']} | {r['validation_mae']:.4f} | {r['validation_rmse']:.4f} | {r['validation_r2']:.4f} | {r['best_epoch']}/{r['epochs_run']} | {r['validation_high_mae']:.4f} | {100*r['validation_high_underprediction_rate']:.1f}% | {r['generalization_gap_mae']:+.4f} | {r['training_seconds']:.1f} |")
base_table=[f"| {r['model']} | {r['mae']:.4f} | {r['rmse']:.4f} | {r['r2']:.4f} |" for r in baseline_rows]
late=[r["variant"] for r in validation_rows if r["late_overfitting_pattern"]]
report=f"""# Phase 2: BiLSTM catalog magnitude regression

Completed on {PROTOCOL['gpu'] or 'CPU'} with corrected v2 dataset hash {dataset_hash}. No prior model weights, v1 arrays, waveform/CNN data or preprocessing reruns.

## Validation-selected configuration
Clean A only: {best_hparams['completed_trials']} completed and {best_hparams['pruned_trials']} pruned TPE trials (15 total), seed 42, SQLite and persisted sampler RNG. Best tuning trial {best_hparams['best_trial']}. Shared parameters: {json.dumps(best_cfg)}.
BiLSTM final forward/backward states are concatenated; dropout and a linear regression head produce one ML value. {winner_row['parameter_count']:,} trainable parameters. AdamW updates every weight and bias; Huber delta 1, clip 1, 50-epoch cap, patience 8 on validation MAE. Targets retain their saved identity transform.

## Controlled validation comparison
All four fits use fresh identical seeded initial weights and selected settings. A clean; B coordinate Gaussian noise (sigma 0.01); C approved regression-relevance complete-window sampling; D weighting plus identical noise. C/D share epoch draws. No interpolation, SMOTE or full SMOGN implementation.

| Variant | MAE | RMSE | R² | Best/run epochs | ML>=4.5 MAE | High underprediction | Val minus clean-train MAE | Fit seconds |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
{chr(10).join(val_table)}

Winner **{winner}** was locked using validation MAE before test-label values were accessed. High-magnitude metrics are tradeoff diagnostics, not an alternative post-hoc selection criterion. Per-epoch unique parents, repeated draws, effective high-magnitude proportions and losses are in variants/*/history.json.

Late-training overfitting pattern (validation MAE rising >0.01 while clean-training MAE falls after its best epoch): {', '.join(late) if late else 'not observed under the predefined criterion'}. For winner {winner}, post-best validation MAE change is {winner_row['post_best_validation_mae_change']:+.4f}; clean-training MAE change {winner_row['post_best_train_mae_change']:+.4f}. Restoring the best checkpoint limits observed late-epoch degradation. A gap alone cannot distinguish overfitting from differences between chronological populations.

## Winner-only test and baselines
| Model | MAE | RMSE | R² |
|---|---:|---:|---:|
{chr(10).join(base_table)}

Test ML>=4.5 count {test_metrics['high_count']}: MAE {test_metrics['high_mae']:.4f}, signed bias {test_metrics['high_bias']:+.4f}, underprediction {100*test_metrics['high_underprediction_rate']:.1f}%. Full magnitude-range errors and residual quantiles are saved. Mean/median baselines use training labels only; recent-event baseline uses the final historical input magnitude with the saved inverse scaler.

Only the selected neural model was evaluated on test. Predictions and selection lock are cached; reruns verify/reuse them and do not retune. Metrics were independently checked with scikit-learn. Source and augmentation hashes remain unchanged.

## Compute, reproducibility and limits
Tuning fit compute: {tuning_seconds/60:.2f} minutes; four controlled fits: {controlled_seconds/60:.2f} minutes; total { (tuning_seconds+controlled_seconds)/60:.2f} minutes. Times include epoch preprocessing, training, validation and best-checkpoint saving; exclude notebook setup and some resume-checkpoint I/O. CUDA deterministic algorithms and TF32-off settings are recorded. All best checkpoints reload exactly on validation; clean refit reproduces the selected tuning trial. Same-seed comparison is not evidence of robustness across random seeds. Selection on a reused validation set can be optimistic.

This is next-retained-ML-catalog-event regression on retrospective historical context, not operational EEW or exact earthquake prediction. Reporting/revision availability and magnitude calibration remain UNKNOWN. Earlier observed targets can become later historical inputs; target splits are disjoint, but overlapping histories are not independent observations. Chronological population differences and source completeness may affect generalization; this experiment does not establish their causes.

## Research scope
[Graves & Schmidhuber (2005)](https://www.cs.toronto.edu/~graves/nn_2005.pdf), Sections II–III: bidirectional finite-segment processing and full-gradient BPTT motivate the architecture. PyTorch cells/head are implementation choices, not an exact reproduction.
[Chen & Fang (2025)](https://doi.org/10.1061/NHREFO.NHENG-2244): catalog sliding-window background from the accessible abstract; full methodology unavailable.
[Al Banna et al. (2021), including Andersson](https://doi.org/10.1109/ACCESS.2021.3071400): catalog seismicity-modeling background; different target/protocol, no exact reproduction.
[SMOTE (2002)](https://doi.org/10.1613/jair.953) and [SMOGN (2017)](https://proceedings.mlr.press/v74/branco17a.html): imbalance motivation only. Approved weighted-window/noise consumers are alternatives, not either algorithm.

## Artifacts
Notebook: ../phase2_bilstm_training.ipynb. Checkpoints/history: variants/A–D/best.pt and history.json. Hyperparameters: best_hyperparameters.json. Study: optuna_study.sqlite3; sampler state: optuna_sampler.pkl. Validation: validation_comparison.csv. Selection: selection_lock.json. Test: selected_test_metrics.json, selected_test_predictions.npz, test_baseline_comparison.csv, test_error_by_magnitude.csv. Figures: plots/. Verification: verification.json.
"""
(OUT/"final_report.md").write_text(report,encoding="utf-8")
def update_notes(path):
    marker="<!-- PHASE2_CURRENT_START -->";end="<!-- PHASE2_CURRENT_END -->"
    old=path.read_text(encoding="utf-8") if path.exists() else ""
    if marker in old and end in old:old=old.split(end,1)[1].lstrip()
    body=f"""# Phase 2 COMPLETE - corrected catalog BiLSTM (2026-11-09)
- Only approved v2 data; run {RUN_ID}. Fifteen clean-A TPE trials ({best_hparams['completed_trials']} complete, {best_hparams['pruned_trials']} pruned), SQLite/sampler/epoch resume saved.
- A/B/C/D trained independently with identical selected settings and fresh seeded initial weights. Winner {winner} selected by validation MAE {winner_row['validation_mae']:.4f}.
- Winner-only test: MAE {test_metrics['mae']:.4f}, RMSE {test_metrics['rmse']:.4f}, R2 {test_metrics['r2']:.4f}; high-magnitude underprediction {100*test_metrics['high_underprediction_rate']:.1f}%.
- Notebook phase2_bilstm_training.ipynb; report phase2_bilstm_results/final_report.md. Checkpoints variants/A-D/best.pt; selection lock, caches and independent verification saved.
- Total fit compute {(tuning_seconds+controlled_seconds)/60:.2f} minutes. No CNN/attention/hybrid processing. Phase 1 artifacts unchanged.
- STOP after Phase 2. No further selection using test. Operational availability/calibration remain UNKNOWN; no EEW capability claim.

## Archived notes
"""
    path.write_text(marker+"\n"+body+end+"\n\n"+old,encoding="utf-8")
update_notes(ROOT/"project_status.md");update_notes(ROOT/".agent-memory/checkpoint.md")
save_json(OUT/"progress.json",{"stage":"COMPLETE","winner":winner,"run_id":RUN_ID})
save_json(OUT/"artifact_hashes.json",{"run_id":RUN_ID,"files":{str(p.relative_to(OUT)):sha(p)
 for p in [OUT/"protocol.json",OUT/"best_hyperparameters.json",OUT/"validation_comparison.csv",OUT/"selection_lock.json",
           test_metrics_path,test_cache,OUT/"verification.json",OUT/"final_report.md",
           *[OUT/"variants"/v/"best.pt" for v in "ABCD"]]}})
print("Phase 2 complete. Winner",winner,"; test MAE",round(test_metrics["mae"],4),"; fit minutes",round((tuning_seconds+controlled_seconds)/60,2))


Phase 2 complete. Winner C ; test MAE 0.5477 ; fit minutes 3.14
